In [1]:
#### BASELINE PLAYER KEY ####
# based on historical football-reference data
# includes draft data from 2025

# import packages
import os
import sys
import pandas as pd
import re
import json

# import local scripts
sys.path.append(os.path.dirname(os.path.abspath('')))
from scripts.load_data import load_data
from scripts.clean_cols import football_ref_cols, football_ref_draft_cols

# Get list of files starting with 's2' in the directory
data_dir = "../data/fpts historical"
files = [f for f in os.listdir(data_dir) if f.startswith('s2')]

# Initialize empty list to store dataframes
dfs = []

# Process each file
for file in files:
    # Extract year from filename (assuming format s2XXX where XXX is year)
    year = file[1:5]
    
    # Load the data
    df = load_data(os.path.join(data_dir, file))

    # drop the first 2 rows
    df = df.iloc[1:]
    
    # Update column names to match football_ref_cols
    df.columns = football_ref_cols
    
    # Add SEASON column
    df['SEASON'] = year
    
    # Clean player names - remove special characters
    df['PLAYER NAME'] = df['PLAYER NAME'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', str(x)))
    
    dfs.append(df)

# Concatenate all dataframes
combined_df = pd.concat(dfs, ignore_index=True)

# Read in draft_2025.csv
draft_2025_df = pd.read_csv(data_dir + '/draft_2025.csv')

# Update column names to match football_ref_draft_cols
draft_2025_df.columns = football_ref_draft_cols

# Subset to only 'PLAYER NAME' and 'PLAYER ID', drop duplicates
draft_2025_alias_df = draft_2025_df[['PLAYER NAME', 'PLAYER ID']].drop_duplicates()

# Subset combined_df to only include 'PLAYER NAME' and 'PLAYER ID' columns
alias_df = combined_df[['PLAYER NAME', 'PLAYER ID']].drop_duplicates()
alias_df = pd.concat([alias_df, draft_2025_alias_df])

# Create player name dictionary
player_dict = alias_df.drop_duplicates('PLAYER ID').set_index('PLAYER ID')['PLAYER NAME'].to_dict()

# save player_dict to json
with open('player_key_dict_base.json', 'w') as f:
    json.dump(player_dict, f, indent=4)





In [2]:
#### FULL PLAYER KEY ####
# includes fantasy rankings and aliases from multiple sources 

# Load each file in data_path into a distinct DataFrame using load_data.

data_path = "../data/rankings current/update/"

# List all files in the data_path directory
files = [f for f in os.listdir(data_path) if not f.startswith('.')]

# Sort files for reproducibility (optional, but helps with mapping)
files.sort()

# Load each file into a DataFrame using load_data
# Step 1: Create a lookup table mapping comment names to generalized file names.
# We'll use the first 12 characters of each file name (adjust as needed for your files).
# The keys are the comment names, the values are the generalized file names.

lookup_keys = [
    "fpts",
    "fantasypros",
    "jj",
    "draftshark_adp",
    "draftshark_rank",
    "hayden_winks"
]

# Generalize file names by taking the first 12 characters (adjust if needed)
generalized_files = [f[:11] for f in files[:len(lookup_keys)]]

# Build the lookup dictionary
file_lookup = dict(zip(lookup_keys, generalized_files))

# Step 2: Use the lookup to read in each file and assign to a variable named after the key.
# We'll use globals() to dynamically assign variable names.

# Create a dictionary of DataFrames, keyed by lookup_key, with values from load_data
dataframes = {}
for key, gen_file in file_lookup.items():
    # Find the actual file in files that starts with gen_file
    matched_file = next((f for f in files if f.startswith(gen_file)), None)
    if matched_file:
        dataframes[key] = load_data(os.path.join(data_path, matched_file))
    else:
        raise ValueError(f"No file found for key '{key}' with prefix '{gen_file}'.")

# Now you have a dictionary: dataframes['fpts'], dataframes['fantasypros'], etc.

In [3]:
# update col names in dataframes dict

from scripts.clean_cols import cols_dict

for key, df in dataframes.items():
    if key in cols_dict:
        df.columns = cols_dict[key]


In [4]:
import re

# 1. Remove special characters from PLAYER NAME in each DataFrame
for df in dataframes.values():
    if 'PLAYER NAME' in df.columns:
        # Remove +, *, ., ', and any other non-alphanumeric (except space)
        df['PLAYER NAME'] = df['PLAYER NAME'].astype(str).apply(lambda x: re.sub(r"[^\w\s]", "", x))

# 2. Create a list of all PLAYER NAME values from all DataFrames
all_player_names = []
for df in dataframes.values():
    if 'PLAYER NAME' in df.columns:
        all_player_names.extend(df['PLAYER NAME'].dropna().tolist())

# 3. Drop duplicates to ensure unique names
unique_player_names = list(set(all_player_names))

# Explain: 
# - We clean PLAYER NAME columns in-place for all DataFrames.
# - We gather all names into a single list, then use set() to deduplicate.
# - unique_player_names now contains all unique, cleaned player names across all DataFrames.


In [5]:
unique_player_names

['Tampa Bay Buccaneers',
 'Derius Davis',
 'Justin Jefferson',
 'Cleveland Browns',
 'Tutu Atwell',
 'DJ Williams',
 'Charlie Jones',
 'Kyler Murray',
 'Joshua Palmer',
 'Jonathan Mingo',
 'Joe Mixon',
 'Josh Whyle',
 'Justin Herbert',
 'Charlie Kolar',
 'Nate Adkins',
 'Geno Smith',
 'Theo Johnson',
 'Jamaal Williams',
 'Jalen Tolbert',
 'Aaron Jones',
 'Jaylin Noel',
 'Bo Melton',
 'Terrell Jennings',
 'Travis Homer',
 'Travis Etienne',
 'Julius Chestnut',
 'Nikko Remigio',
 'Mike Gesicki',
 'Brevyn SpannFord',
 'Ryan Flournoy',
 'Trevor Etienne',
 'Cameron Ward',
 'Jalen Nailor',
 'Rondale Moore',
 'Roschon Johnson',
 'Tyler Conklin',
 'Aaron Rodgers',
 'Will Levis',
 'Bryce Young',
 'Denver Broncos',
 'Cam Ward',
 'Ollie Gordon II',
 'Caleb Lohner',
 'Christian McCaffrey',
 'Michael Pittman',
 'Eddy Pineiro',
 'CJ Stroud',
 'Tyler Warren',
 'Quinshon Judkins',
 'Tua Tagovailoa',
 'Chimere Dike',
 'Mason Tipton',
 'Darnell Mooney',
 'Noah Fant',
 'David Njoku',
 'Patrick Taylor',
 '

In [6]:
from difflib import SequenceMatcher
import json

def match_players_to_keys(unique_player_names, player_key_dict, threshold=0.8):
    """
    Matches player names to existing keys in player_key_dict based on similarity scores.
    
    Args:
        unique_player_names (list): List of unique player names to match
        player_key_dict (dict): Dictionary with keys and string player names as values
        threshold (float): Minimum similarity score for a match (default 0.8)
    
    Returns:
        dict: Updated player_key_dict with new matches (converted to lists)
        dict: Stats about matching (matched_count, total_count, unmatched_players)
    """
    matched_count = 0
    unmatched_players = []
    updated_dict = {}
    
    # Convert all dictionary values to lists if they aren't already
    for key, value in player_key_dict.items():
        if isinstance(value, list):
            updated_dict[key] = value.copy()
        else:
            updated_dict[key] = [value]
    
    # Explain: For each unique player name, find the best matching key based on similarity
    for player_name in unique_player_names:
        best_match_key = None
        best_score = 0
        
        # Explain: Compare against all existing player names in the dictionary values
        for key, existing_name in player_key_dict.items():
            # Handle both string and list values in the original dictionary
            if isinstance(existing_name, list):
                names_to_check = existing_name
            else:
                names_to_check = [existing_name]
            
            for name in names_to_check:
                # Calculate similarity score using SequenceMatcher
                score = SequenceMatcher(None, player_name.lower(), name.lower()).ratio()
                
                if score > best_score:
                    best_score = score
                    best_match_key = key
        
        # Explain: If similarity score exceeds threshold, add to the key's value list
        if best_score >= threshold and best_match_key:
            if player_name not in updated_dict[best_match_key]:
                updated_dict[best_match_key].append(player_name)
            matched_count += 1
        else:
            unmatched_players.append(player_name)
    
    # Explain: Compile statistics about the matching process
    stats = {
        'matched_count': matched_count,
        'total_count': len(unique_player_names),
        'unmatched_count': len(unmatched_players),
        'match_rate': matched_count / len(unique_player_names) if unique_player_names else 0,
        'unmatched_players': unmatched_players
    }
    
    return updated_dict, stats

# Load the existing player key dictionary
with open('player_key_dict_base.json', 'r') as f:
    player_key_dict = json.load(f)

# Run the matching function
updated_player_key_dict, matching_stats = match_players_to_keys(unique_player_names, player_key_dict)

# Report results
print(f"Matching Results:")
print(f"Total players: {matching_stats['total_count']}")
print(f"Matched players: {matching_stats['matched_count']}")
print(f"Unmatched players: {matching_stats['unmatched_count']}")
print(f"Match rate: {matching_stats['match_rate']:.2%}")

if matching_stats['unmatched_players']:
    print(f"\nUnmatched players ({len(matching_stats['unmatched_players'])}):")
    for player in matching_stats['unmatched_players'][:20]:  # Show first 20
        print(f"  - {player}")
    if len(matching_stats['unmatched_players']) > 20:
        print(f"  ... and {len(matching_stats['unmatched_players']) - 20} more")


# save the updated player key dictionary
with open('player_key_dict.json', 'w') as f:
    json.dump(updated_player_key_dict, f, indent=4)

KeyboardInterrupt: 

In [22]:
'Hollywood Brown'
'JJ McCarthy'
'Xavier Restrepo'

'Xavier Restrepo'

In [23]:
updated_player_key_dict

{'McCaCh01': ['Christian McCaffrey'],
 'JackLa00': ['Lamar Jackson'],
 'HenrDe00': ['Derrick Henry'],
 'JoneAa00': ['Aaron Jones', 'Aaron Jones Sr'],
 'ElliEz00': ['Ezekiel Elliott'],
 'CookDa01': ['Dalvin Cook'],
 'ThomMi05': ['Michael Thomas'],
 'KelcTr00': ['Travis Kelce'],
 'ChubNi00': ['Nick Chubb'],
 'IngrMa01': ['Mark Ingram'],
 'EkelAu00': ['Austin Ekeler'],
 'PresDa01': ['Dak Prescott'],
 'AndrMa00': ['Mark Andrews'],
 'WilsRu00': ['Russell Wilson'],
 'GodwCh00': ['Chris Godwin'],
 'KittGe00': ['George Kittle'],
 'WatsDe00': ['Deshaun Watson'],
 'CarsCh00': ['Chris Carson'],
 'GollKe00': ['Kenny Golladay'],
 'WallDa01': ['Darren Waller'],
 'BarkSa00': ['Saquon Barkley'],
 'ErtzZa00': ['Zach Ertz'],
 'MixoJo00': ['Joe Mixon'],
 'GurlTo01': ['Todd Gurley'],
 'CookJa02': ['Jared Cook'],
 'KuppCo00': ['Cooper Kupp'],
 'JoneJu02': ['Julio Jones'],
 'ParkDe01': ['DeVante Parker'],
 'FourLe00': ['Leonard Fournette'],
 'WinsJa00': ['Jameis Winston'],
 'HoopAu00': ['Austin Hooper'],
 '